In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaLLM 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ૧. કાલ્પનિક ડેટા ફાઈલ (તમારો પ્રાઈવેટ ડેટા સેવ કરવો)
data = """
Company Rules 2026:
- Office timings are from 9:00 AM to 6:00 PM.
- Saturday and Sunday will be official holidays.
- Employees will get a total of 15 paid leaves per year.
- Free lunch is provided to all employees at 1:00 PM.
- Formal dress code must be followed from Monday to Thursday.
"""
with open("company_rules.txt", "w", encoding="utf-8") as f:
    f.write(data)

# ૨. ડેટા લોડ કરવો અને તેના નાના ટુકડા (Chunks) કરવા
loader = TextLoader("company_rules.txt", encoding="utf-8")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)

# ૩. ફ્રી લોકલ એમ્બેડિંગ મોડેલ સેટ કરવું (ઇન્ટરનેટ વગર ચાલશે)
model_name = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# ૪. ડેટાને ChromaDB વેક્ટર ડેટાબેઝમાં સ્ટોર કરવો અને રીટ્રીવર સેટ કરવું
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# ૫. લોકલ Ollama મોડેલ (Llama 3.2 1B) કનેક્ટ કરવું
llm = OllamaLLM(model="llama3.2")

# ૬. પ્રોમ્પ્ટ ટેમ્પલેટ સેટઅપ (AI ને કેવી રીતે જવાબ આપવો તે કહેવા માટે)
template = """Answer the question based only on the following context:
Context: {context}

Question: {question}
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# ૭. LCEL પદ્ધતિથી આખી RAG પાઇપલાઇન ચેઇન બનાવવી
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ૮. ટેસ્ટિંગ માટે ૫ અલગ-અલગ પ્રશ્નોની યાદી (Five Questions List)
questions = [
    "How many paid leaves do employees get per year?",
    "What are the office timings?",
    "Which days are official holidays?",
    "At what time is free lunch provided?",
    "What is the dress code rule for Monday to Thursday?"
]

# ૯. લૂપ ચલાવીને બધા પ્રશ્નોના જવાબો મેળવવા
print("--- Starting RAG Testing With 5 Questions ---\n")
for i, query in enumerate(questions, 1):
    print(f"Question {i}: {query}")
    response = rag_chain.invoke(query)
    print(f"Answer: {response}")
    print("-" * 50)